# 00 · Chạy thử baseline trên CPU

Dữ liệu **tự tạo**, 3 nhãn minh họa; không phải benchmark GoEmotions (28 nhãn).
Không tải dataset/checkpoint. Dùng đúng pipeline của `scripts/baseline/run_baseline.py` và metric chung. Xem `README.md#baseline`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from scripts.baseline.run_baseline import build_model
from src.evaluation.metrics import evaluate_multilabel

## 1. Văn bản và multi-hot

Mỗi hàng có thể có nhiều giá trị 1. Tập validation dưới đây là các câu mới; TF-IDF chỉ học từ train.

In [ ]:
labels = ['gratitude', 'joy', 'sadness']
train_texts = ['thank you happy', 'thank you friend', 'happy wonderful', 'happy today',
               'sad alone', 'sad today', 'thank you wonderful', 'happy thank you']
y_train = np.array([[1, 1, 0], [1, 0, 0], [0, 1, 0], [0, 1, 0],
                    [0, 0, 1], [0, 0, 1], [1, 0, 0], [1, 1, 0]])
val_texts = ['thank you', 'happy', 'sad', 'happy thank you']
y_val = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 0]])
pd.DataFrame(y_train, columns=labels).assign(text=train_texts)

## 2. Huấn luyện

Gọi cùng `build_model` của baseline thật; số LR bằng số cột nhãn. Trong GoEmotions sẽ là 28, trong ví dụ này là 3. Dùng balanced để minh họa trọng số lớp.

In [ ]:
model = build_model('balanced')
model.fit(train_texts, y_train)
print('Vocabulary chỉ học từ train:', model.named_steps['tfidf'].get_feature_names_out())
print('Số LR:', len(model.named_steps['classifier'].estimators_))

## 3. Điểm dự đoán và ngưỡng

`predict_proba` trả một điểm độc lập cho mỗi nhãn. Không dùng argmax. Ngưỡng 0.5 ở đây là cố định; notebook không tuning bằng tập validation minh họa.

In [ ]:
scores = model.predict_proba(val_texts)
threshold = 0.5
predicted = scores >= threshold
view = pd.DataFrame(scores, columns=labels)
view['text'] = val_texts
view['predicted_labels'] = [[label for label, yes in zip(labels, row) if yes] for row in predicted]
view

## 4. Đánh giá bằng module chung

Các số dưới đây chỉ kiểm luồng code trên dữ liệu tự tạo. Không đưa vào bảng kết quả đồ án.

In [ ]:
metrics = evaluate_multilabel(y_val, scores, labels, threshold=threshold)
pd.Series({name: metrics[name] for name in
           ['macro_f1', 'micro_f1', 'micro_precision', 'micro_recall', 'hamming_loss']})

## 5. Tự thử một câu

Đổi câu tiếng Anh bên dưới rồi chạy lại. Với câu chứa từ ngoài vocabulary, mô hình không học được ý nghĩa mới từ câu đó.

In [ ]:
text = 'happy thank you friend'
row = model.predict_proba([text])[0]
print(text)
print({label: round(float(score), 4) for label, score in zip(labels, row)})
print('Nhãn:', [label for label, score in zip(labels, row) if score >= threshold])

## 6. Chuyển sang GoEmotions

Đọc README và notebook 01/02. Chạy hai model full và analyze trước notebook 02. Train 43.410 / val 5.426 / test 5.427, giữ 28 nhãn. Khóa cấu hình/ngưỡng trước test. Notebook này không lưu đè model hoặc kết quả benchmark.